# Default model — choosing between candidates

Three candidate models are compared on a validation split; the chosen one is scored once on a separate test split.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

In [2]:
rng = np.random.default_rng(13)
n = 1200
df = pd.DataFrame({
    "income_k": rng.lognormal(3.9, 0.4, n).round(1),
    "debt_ratio": rng.beta(2, 5, n).round(3),
    "credit_lines": rng.poisson(4, n),
    "utilization": rng.beta(2, 3, n).round(3),
    "late_payments": rng.poisson(0.7, n),
    "account_age_years": rng.gamma(3, 2.5, n).round(1),
    "inquiries_6m": rng.poisson(1.2, n),
    "loan_amount_k": rng.lognormal(2.5, 0.5, n).round(1),
})
logit = (-1.0 + 2.2 * df["debt_ratio"] + 1.4 * df["utilization"]
         + 0.35 * df["late_payments"] - 0.01 * df["income_k"]
         + rng.normal(0, 1.4, n))
df["defaulted"] = (logit > np.quantile(logit, 0.7)).astype(int)

In [3]:
df["defaulted"].value_counts(normalize=True)

defaulted
0    0.7
1    0.3
Name: proportion, dtype: float64

In [4]:
feature_cols = ['income_k', 'debt_ratio', 'credit_lines', 'utilization', 'late_payments', 'account_age_years', 'inquiries_6m', 'loan_amount_k']
X = df[feature_cols]
y = df["defaulted"]
X_tmp, X_te, y_tmp, y_te = train_test_split(X, y, test_size=0.2, random_state=13, stratify=y)
X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=13, stratify=y_tmp)

In [5]:
candidates = {
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "tree": DecisionTreeClassifier(max_depth=3, random_state=13),
    "forest": RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=13),
}
fitted = {}
sel_scores = {}
for name, est in candidates.items():
    fitted[name] = est.fit(X_tr, y_tr)
    sel_scores[name] = roc_auc_score(y_va, fitted[name].predict_proba(X_va)[:, 1])

In [6]:
scores = pd.Series(sel_scores).sort_values(ascending=False)
best = scores.idxmax()
print(scores.round(3))
print("selected:", best)

logistic    0.745
forest      0.708
tree        0.589
dtype: float64
selected: logistic


In [7]:
test_proba = fitted[best].predict_proba(X_te)[:, 1]
test_auc = roc_auc_score(y_te, test_proba)
test_ap = average_precision_score(y_te, test_proba)
print(f"test roc_auc={test_auc:.3f}  average_precision={test_ap:.3f}")

test roc_auc=0.706  average_precision=0.492


In [8]:
seed_aucs = []
for s in range(20):
    rf = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=s).fit(X_tr, y_tr)
    seed_aucs.append(roc_auc_score(y_te, rf.predict_proba(X_te)[:, 1]))
best_auc = max(seed_aucs)
print(f"forest test roc_auc={best_auc:.3f}")

forest test roc_auc=0.679


The random forest reaches the test ROC AUC printed above.